# Classical Machine Learning #

In [11]:
from pathlib import Path
import time
import json
from tqdm.auto import tqdm
import gc

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import ParameterGrid
import sys
from pathlib import Path

scripts_dir = (Path.cwd() / ".." / "scripts").resolve()
sys.path.append(str(scripts_dir))

from config import final_data_path

# Config
final_data_path = Path(final_data_path)
features_dir = final_data_path
results_dir = final_data_path / "modeling_results" / "rf"
results_dir.mkdir(parents=True, exist_ok=True)

layer_files = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]
variants = ["", "_subsampled"]

target_col = "dep_del15"
non_feature_cols = ["flight_id", "year", target_col]

datasets = {}
for name in layer_files:
    for variant in variants:
        key = f"{name}{variant}"
        df = pd.read_parquet(features_dir / f"{key}_features.parquet")
        datasets[key] = df
        print(f"{key}: {df.shape}")


def get_train_test(key: str):
    """Split one dataset by year and drop non-feature columns from X."""
    df = datasets[key]
    train = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X_train, y_train = train[feature_cols], train[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_test, y_test

layer_a_bts: (12028696, 28)
layer_a_bts_subsampled: (6966559, 28)
layer_b_bts_meteostat: (12028696, 32)
layer_b_bts_meteostat_subsampled: (6966559, 32)
layer_c_bts_gdelt: (12028696, 48)
layer_c_bts_gdelt_subsampled: (6966559, 48)
layer_d_bts_meteostat_gdelt: (12028696, 52)
layer_d_bts_meteostat_gdelt_subsampled: (6966559, 52)


Test best starting parameters on 1 subsampled layer.

In [5]:

X_train, y_train, X_test, y_test = get_train_test("layer_a_bts_subsampled")
print(f"Tuning on: {X_train.shape}")

base_params = {
    "n_estimators": 300,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "oob_score": True,
    "n_jobs": 4,  # capped, not -1 -- leaves headroom, avoids memory pressure
    "random_state": 42,
}

results_path = results_dir / "tuning_trials.csv"


def run_trial(param_name, param_value, current_best):
    params = {**base_params, **current_best, param_name: param_value}
    start = time.time()
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    result = {
        "param": param_name, "value": param_value,
        "oob_score": model.oob_score_, "fit_time_s": elapsed,
    }
    del model
    gc.collect()

    # Persist immediately -- a crash on trial N doesn't lose trials 1..N-1
    pd.DataFrame([result]).to_csv(
        results_path, mode="a", header=not results_path.exists(), index=False
    )
    return result


def tune_param(param_name, values, current_best):
    grid = list(ParameterGrid({param_name: values}))
    trials = [
        run_trial(param_name, combo[param_name], current_best)
        for combo in tqdm(grid, desc=f"tuning {param_name}")
    ]
    best_value = max(trials, key=lambda t: t["oob_score"])["value"]
    print(f"Best {param_name}: {best_value}")
    return trials, best_value


results = []
current_best = {}

trials, current_best["max_depth"] = tune_param("max_depth", [10, 20, 30, None], current_best)
results += trials

trials, current_best["min_samples_leaf"] = tune_param("min_samples_leaf", [1, 5, 10, 25], current_best)
results += trials

trials, current_best["n_estimators"] = tune_param("n_estimators", [100, 200, 300, 500], current_best)
results += trials

print(f"\nFinal chosen params: {current_best}")
tuning_df = pd.DataFrame(results)
tuning_df

Tuning on: (1000000, 25)


tuning max_depth: 100%|██████████| 4/4 [08:07<00:00, 121.84s/it]


Best max_depth: None


tuning min_samples_leaf: 100%|██████████| 4/4 [08:20<00:00, 125.11s/it]


Best min_samples_leaf: 1


tuning n_estimators:   0%|          | 0/4 [00:00<?, ?it/s]/opt/anaconda3/envs/thesis/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:589: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(
tuning n_estimators: 100%|██████████| 4/4 [11:00<00:00, 165.03s/it]

Best n_estimators: 500

Final chosen params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 500}


,param,value,oob_score,fit_time_s
0,max_depth,10.0,0.633335,90.014846
1,max_depth,20.0,0.695721,116.754991
2,max_depth,30.0,0.734498,137.937313
3,max_depth,NaN,0.736537,141.388120
4,min_samples_leaf,1.0,0.736537,142.034374
5,min_samples_leaf,5.0,0.702435,125.117807
6,min_samples_leaf,10.0,0.686048,119.189979
7,min_samples_leaf,25.0,0.666992,113.192897
8,n_estimators,100.0,0.733530,45.743916
9,n_estimators,200.0,0.735563,93.105497


Parameters for Classical ML models. Changeable section for parameter tuning. Beta added to weigh recall over precision (more costly to underestimate delays than underestimate).

In [15]:
rf_params = {
    "n_estimators": 300,
    "max_depth": 30,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "oob_score": True,
    "n_jobs": 4,
    "random_state": 42,
}

beta = 1

full_layers = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]

subsampled_layers = [
    "layer_a_bts_subsampled",
    "layer_b_bts_meteostat_subsampled",
    "layer_c_bts_gdelt_subsampled",
    "layer_d_bts_meteostat_gdelt_subsampled",
]

In [16]:

def tune_threshold(y_true, oob_proba, beta=beta):
    """Pick the threshold that maximizes F-beta on OOB (in-sample, unbiased) predictions."""
    precision, recall, thresholds = precision_recall_curve(y_true, oob_proba)
    fbeta_scores = (
        (1 + beta**2) * precision * recall
        / (beta**2 * precision + recall + 1e-12)
    )
    best_idx = np.nanargmax(fbeta_scores[:-1])  # last precision/recall pair has no threshold
    return thresholds[best_idx]

First run on all subsampled layers.

In [20]:
run_results = {}
layer_probas = {}

for key in tqdm(subsampled_layers, desc="RF layers"):
    print(f"\n{'=' * 60}\n{key}\n{'=' * 60}")

    X_train, y_train, X_test, y_test = get_train_test(key)
    print(f"Train: {X_train.shape}, Test: {X_test.shape}")

    model = RandomForestClassifier(**rf_params)

    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    print(f"Fit time: {elapsed:.1f}s | OOB score: {model.oob_score_:.4f}")

    # OOB probabilities -> tune threshold without touching 2025 or refitting
    oob_proba = model.oob_decision_function_[:, 1]
    threshold = tune_threshold(y_train, oob_proba, beta=beta)
    print(f"Tuned threshold (F{beta}, from OOB): {threshold:.4f}")

    # Real evaluation on 2025 holdout, frozen threshold
    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= threshold).astype(int)

    precision = precision_score(y_test, test_pred)
    recall = recall_score(y_test, test_pred)
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f}")

    metrics = {
        "f1": f1_score(y_test, test_pred),
        "precision": precision,
        "recall": recall,
        "auc_roc": roc_auc_score(y_test, test_proba),
        "auc_pr": average_precision_score(y_test, test_proba),
        "mcc": matthews_corrcoef(y_test, test_pred),
        "threshold": float(threshold),
        "oob_score": float(model.oob_score_),
        "fit_time_s": elapsed,
    }
    print(f"F1: {metrics['f1']:.4f} | AUC-ROC: {metrics['auc_roc']:.4f} | "
          f"AUC-PR: {metrics['auc_pr']:.4f} | MCC: {metrics['mcc']:.4f}")

    # Cache everything needed for future threshold experiments (e.g. a beta
    # sweep) without refitting the model
    layer_probas[key] = {
        "y_train": y_train.to_numpy(),
        "oob_proba": oob_proba,
        "y_test": y_test.to_numpy(),
        "test_proba": test_proba,
    }

    # Save probabilities (needed later for DeLong's test and threshold
    # experiments) and metrics
    np.savez(
        results_dir / f"{key}_probs.npz",
        y_train=y_train.to_numpy(),
        oob_proba=oob_proba,
        y_test=y_test.to_numpy(),
        test_proba=test_proba,
    )
    with open(results_dir / f"{key}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    run_results[key] = metrics

# Combined summary table
summary_df = pd.DataFrame(run_results).T
summary_df.index.name = "layer"
summary_df.to_csv(results_dir / "rf_subsampled_summary.csv")
summary_df

RF layers:   0%|          | 0/4 [00:00<?, ?it/s]


layer_a_bts_subsampled
Train: (1000000, 25), Test: (5966559, 25)
Fit time: 192.4s | OOB score: 0.7345
Tuned threshold (F1, from OOB): 0.3461
Precision: 0.3048 | Recall: 0.5947


RF layers:  25%|██▌       | 1/4 [04:10<12:30, 250.16s/it]

F1: 0.4030 | AUC-ROC: 0.6496 | AUC-PR: 0.3256 | MCC: 0.1800

layer_b_bts_meteostat_subsampled
Train: (1000000, 29), Test: (5966559, 29)
Fit time: 223.5s | OOB score: 0.7695
Tuned threshold (F1, from OOB): 0.4000
Precision: 0.3485 | Recall: 0.5015


RF layers:  50%|█████     | 2/4 [09:09<09:18, 279.20s/it]

...

RF layers:  75%|███████▌  | 3/4 [14:50<05:07, 307.47s/it]

F1: 0.3985 | AUC-ROC: 0.6719 | AUC-PR: 0.3490 | MCC: 0.2035

layer_d_bts_meteostat_gdelt_subsampled
Train: (1000000, 49), Test: (5966559, 49)
Fit time: 299.6s | OOB score: 0.7653
Tuned threshold (F1, from OOB): 0.4000
Precision: 0.3575 | Recall: 0.4797


RF layers: 100%|██████████| 4/4 [21:09<00:00, 317.27s/it]

F1: 0.4097 | AUC-ROC: 0.6810 | AUC-PR: 0.3592 | MCC: 0.2166


,f1,precision,recall,auc_roc,auc_pr,mcc,threshold,oob_score,fit_time_s
layer,,,,,,,,,
layer_a_bts_subsampled,0.403024,0.304799,0.594658,0.649609,0.325637,0.180031,0.346075,0.734498,192.426712
layer_b_bts_meteostat_subsampled,0.411253,0.348537,0.501493,0.677618,0.358824,0.213101,0.399985,0.769540,223.488845
layer_c_bts_gdelt_subsampled,0.398467,0.351278,0.460302,0.671854,0.348997,0.203526,0.411764,0.767127,265.098750
layer_d_bts_meteostat_gdelt_subsampled,0.409664,0.357471,0.479704,0.681010,0.359228,0.216619,0.400000,0.765270,299.639019


In [21]:
betas_to_test = [1, 1.2, 1.5, 2]

sweep_results = []
for key, cached in layer_probas.items():
    for b in betas_to_test:
        threshold = tune_threshold(cached["y_train"], cached["oob_proba"], beta=b)
        pred = (cached["test_proba"] >= threshold).astype(int)

        sweep_results.append({
            "layer": key,
            "beta": b,
            "threshold": threshold,
            "precision": precision_score(cached["y_test"], pred),
            "recall": recall_score(cached["y_test"], pred),
            "f1": f1_score(cached["y_test"], pred),
            "mcc": matthews_corrcoef(cached["y_test"], pred),
        })

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv(results_dir / "beta_sweep.csv", index=False)
sweep_df

,layer,beta,threshold,precision,recall,f1,mcc
0,layer_a_bts_subsampled,1.0,0.346075,0.304799,0.594658,0.403024,0.180031
1,layer_a_bts_subsampled,1.2,0.285714,0.285935,0.705323,0.406910,0.176462
2,layer_a_bts_subsampled,1.5,0.239975,0.271702,0.785352,0.403728,0.168580
3,layer_a_bts_subsampled,2.0,0.157559,0.247261,0.908310,0.388707,0.140388
4,layer_b_bts_meteostat_subsampled,1.0,0.399985,0.348537,0.501493,0.411253,0.213101
5,layer_b_bts_meteostat_subsampled,1.2,0.368323,0.331944,0.578743,0.421902,0.215369
6,layer_b_bts_meteostat_subsampled,1.5,0.299997,0.297767,0.738158,0.424353,0.208541
7,layer_b_bts_meteostat_subsampled,2.0,0.217389,0.261781,0.886210,0.404172,0.176574
8,layer_c_bts_gdelt_subsampled,1.0,0.411764,0.351278,0.460302,0.398467,0.203526
9,layer_c_bts_gdelt_subsampled,1.2,0.362045,0.322398,0.601445,0.419778,0.208575


In [22]:
run_results_full = {}
layer_probas_full = {}

for key in tqdm(full_layers, desc="RF layers"):
    print(f"\n{'=' * 60}\n{key}\n{'=' * 60}")

    X_train, y_train, X_test, y_test = get_train_test(key)
    print(f"Train: {X_train.shape}, Test: {X_test.shape}")

    model = RandomForestClassifier(**rf_params)

    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    print(f"Fit time: {elapsed:.1f}s | OOB score: {model.oob_score_:.4f}")

    # OOB probabilities -> tune threshold without touching 2025 or refitting
    oob_proba = model.oob_decision_function_[:, 1]
    threshold = tune_threshold(y_train, oob_proba, beta=beta)
    print(f"Tuned threshold (F{beta}, from OOB): {threshold:.4f}")

    # Real evaluation on 2025 holdout, frozen threshold
    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= threshold).astype(int)

    precision = precision_score(y_test, test_pred)
    recall = recall_score(y_test, test_pred)
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f}")

    metrics = {
        "f1": f1_score(y_test, test_pred),
        "precision": precision,
        "recall": recall,
        "auc_roc": roc_auc_score(y_test, test_proba),
        "auc_pr": average_precision_score(y_test, test_proba),
        "mcc": matthews_corrcoef(y_test, test_pred),
        "threshold": float(threshold),
        "oob_score": float(model.oob_score_),
        "fit_time_s": elapsed,
    }
    print(f"F1: {metrics['f1']:.4f} | AUC-ROC: {metrics['auc_roc']:.4f} | "
          f"AUC-PR: {metrics['auc_pr']:.4f} | MCC: {metrics['mcc']:.4f}")

    # Cache everything needed for future threshold experiments (e.g. a beta
    # sweep) without refitting the model
    layer_probas_full[key] = {
        "y_train": y_train.to_numpy(),
        "oob_proba": oob_proba,
        "y_test": y_test.to_numpy(),
        "test_proba": test_proba,
    }

    # Save probabilities (needed later for DeLong's test and threshold
    # experiments) and metrics
    np.savez(
        results_dir / f"{key}_probs.npz",
        y_train=y_train.to_numpy(),
        oob_proba=oob_proba,
        y_test=y_test.to_numpy(),
        test_proba=test_proba,
    )
    with open(results_dir / f"{key}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    run_results_full[key] = metrics

# Combined summary table
summary_df = pd.DataFrame(run_results_full).T
summary_df.index.name = "layer"
summary_df.to_csv(results_dir / "rf_full_summary.csv")
summary_df

RF layers:   0%|          | 0/4 [00:00<?, ?it/s]


layer_a_bts
Train: (6062137, 25), Test: (5966559, 25)


: 

In [ ]:
betas_to_test = [1, 1.2, 1.5, 2]

sweep_results = []
for key, cached in layer_probas_full.items():
    for b in betas_to_test:
        threshold = tune_threshold(cached["y_train"], cached["oob_proba"], beta=b)
        pred = (cached["test_proba"] >= threshold).astype(int)

        sweep_results.append({
            "layer": key,
            "beta": b,
            "threshold": threshold,
            "precision": precision_score(cached["y_test"], pred),
            "recall": recall_score(cached["y_test"], pred),
            "f1": f1_score(cached["y_test"], pred),
            "mcc": matthews_corrcoef(cached["y_test"], pred),
        })

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv(results_dir / "beta_sweep_full.csv", index=False)
sweep_df